In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# مقدمة إلى الأوليات (Primitives)

<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</AccordionItem>
</Accordion>
<span id="qpu-access-patterns"></span>
## لماذا أدخل Qiskit الأوليات؟
على غرار الأيام الأولى للحواسيب الكلاسيكية، حين كان على المطورين التعامل مع سجلات المعالج مباشرةً، كانت الواجهة الأولى للـ QPUs تُعيد البيانات الخام من الإلكترونيات الضابطة.
لم يكن هذا مشكلة كبيرة حين كانت الـ QPUs في المختبرات ولا يصل إليها إلا الباحثون مباشرةً.
إدراكًا بأن معظم المطورين لن يكونوا ملمّين — ولا ينبغي أن يكونوا — بكيفية تحليل هذه البيانات الخام إلى أصفار وآحاد، أدخل Qiskit الدالة `backend.run`، وهي أول مستوى تجريد للوصول إلى الـ QPUs عبر الحوسبة السحابية. أتاح ذلك للمطورين العمل بصيغ بيانات مألوفة والتركيز على الصورة الأكبر.

مع اتساع الوصول إلى الـ QPUs وتطور المزيد من الخوارزميات الكمية، برزت مجددًا الحاجة إلى مستوى تجريد أعلى. استجابةً لذلك، أدخل Qiskit واجهة الأوليات، المُحسَّنة لمهمتين أساسيتين في تطوير الخوارزميات الكمية: تقدير قيم التوقع (`Estimator`) وأخذ عينات من الدوائر (`Sampler`). الهدف مرة أخرى هو مساعدة المطورين على التركيز أكثر على الابتكار وأقل على تحويل البيانات. تحل واجهة الأوليات محل واجهة `backend.run`، إذ يوفر `Sampler` نفس الوصول المباشر إلى العتاد الذي كانت تتيحه `backend.run`.

## ما هي الأولية (Primitive)؟
تُبنى أنظمة الحوسبة على طبقات متعددة من التجريد. تتيح لك التجريدات التركيز على مستوى معين من التفاصيل ذات الصلة بالمهمة التي بين يديك. كلما اقتربت من العتاد، قلّ مستوى التجريد الذي تحتاجه (مثلًا، قد تحتاج إلى نقل البيانات أو معالجتها على مستوى تعليمات المعالج). وكلما تعقّدت المهمة التي تريد تنفيذها، ارتفع مستوى التجريدات المستخدمة (مثلًا، قد تستعين بمكتبة برمجية لإجراء حسابات جبرية).

في هذا السياق، *الأولية* هي أصغر وحدة معالجة، اللبنة الأساسية الأبسط التي يمكن من خلالها بناء شيء مفيد لمستوى تجريد معين.

زاد التقدم الأخير في الحوسبة الكمية من الحاجة للعمل عند مستويات تجريد أعلى.
مع تحرك هذا المجال نحو وحدات معالجة كمية (QPUs) أكبر وسير عمل أكثر تعقيدًا، يتحوّل التركيز من التفاعل مع إشارات الكيوبت الفردية إلى النظر إلى الأجهزة الكمية كأنظمة تؤدي المهام المطلوبة.

أكثر مهمتين شيوعًا في الحواسيب الكمية هما: أخذ عينات من الحالات الكمية وحساب قيم التوقع.
هاتان المهمتان شكّلتا الدافع وراء تصميم أوليات Qiskit: **Estimator** و**Sampler**.

- يحسب Estimator قيم توقع المقاييس (observables) بالنسبة إلى الحالات التي تُهيئها الدوائر الكمية.
- يأخذ Sampler عينات من سجل الإخراج الناتج عن تشغيل الدوائر الكمية.

باختصار، يُقرّب النموذج الحسابي الذي تقدمه أوليات Qiskit البرمجة الكمية خطوةً من حيث وصلت إليه البرمجة الكلاسيكية اليوم، حيث يقل التركيز على تفاصيل العتاد ويزيد على النتائج التي تسعى إلى تحقيقها.

## تعريف الأوليات وتطبيقاتها
يوجد نوعان من أوليات Qiskit: الفئات الأساسية (base classes)، وتطبيقاتها (implementations). تُعرَّف أوليتا Estimator وSampler عبر فئات أساسية مفتوحة المصدر موجودة في حزمة Qiskit SDK (في وحدة [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives)). يمكن للمزودين (مثل Qiskit Runtime) استخدام هذه الفئات الأساسية لاشتقاق تطبيقاتهم الخاصة من Sampler وEstimator. سيتعامل معظم المستخدمين مع تطبيقات المزودين، لا مع الأوليات الأساسية مباشرةً.

### الفئات الأساسية
الأوليات `Base` هي فئات مجردة تُعرّف واجهة مشتركة لتطبيق الأوليات. جميع الفئات الأخرى في وحدة [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives) ترث من هذه الفئات الأساسية. ينبغي للمطورين استخدامها إن كانوا مهتمين بإنشاء نموذج تنفيذ خاص بهم يعتمد على الأوليات لمزود محدد. قد تفيد هذه الفئات أيضًا من يريد إجراء معالجة مخصصة للغاية ويجد أن التطبيقات الحالية للأوليات بسيطة جدًا لاحتياجاته. لن يستخدم المستخدمون العاديون الفئات الأساسية مباشرةً.

[`BaseEstimatorV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV1) و[`BaseSamplerV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV1) - رغم أن أوليات V1 لا تزال قابلة للاستخدام، تركّز هذه الأدلة على أوليات V2 لأنها الأحدث والأكثر شيوعًا.

[`BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2) و[`BaseSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV2) - تتبع الأوليات المرجعية في Qiskit مواصفات هذه الواجهات.

<span id="implementations"></span>
### التطبيقات
تُنشأ جميع الأوليات من الفئات الأساسية؛ لذا فإنها تشترك في نفس البنية العامة وطريقة الاستخدام. مثلًا، صيغة المدخلات لجميع أوليات Estimator واحدة. غير أن هناك اختلافات في التطبيقات تجعل كلًا منها فريدًا.

هذه هي تطبيقات الفئات الأساسية للأوليات:

- توفر [أوليات Qiskit Runtime](/guides/qiskit-runtime-primitives)، [`EstimatorV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) و[`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2)، تطبيقًا أكثر تطورًا (مثلًا، بتضمين تخفيف الأخطاء) كخدمة سحابية. يُستخدم هذا التطبيق للوصول إلى عتاد IBM Quantum&reg;.

- [`StatevectorEstimator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorEstimator) و[`StatevectorSampler`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorSampler#statevectorsampler) - تطبيقات مرجعية للأوليات تستخدم المحاكي المدمج في Qiskit. بُنيت باستخدام وحدة Qiskit [`quantum_info`](https://docs.quantum.ibm.com/api/qiskit/quantum_info#quantum-information)، وتنتج نتائج مبنية على محاكاة متجه الحالة المثالية. يُوصَل إليها عبر Qiskit. راجع [المحاكاة الدقيقة باستخدام أوليات Qiskit](/guides/simulate-with-qiskit-sdk-primitives) للاطلاع على تفاصيل الاستخدام.

- [`BackendEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendEstimatorV2) و[`BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) - يمكنك استخدام هذه الفئات "لتغليف" أي مورد حوسبة كمية في أولية. يتيح لك ذلك كتابة كود بأسلوب الأوليات لمزودين لا يملكون بعد واجهة مبنية على الأوليات. يمكن استخدام هذه الفئات تمامًا مثل Sampler وEstimator العاديين، باستثناء أنه ينبغي تهيئتها بمعامل إضافي `backend` لتحديد الحاسوب الكمي المُراد التشغيل عليه. يُوصَل إليها باستخدام Qiskit. راجع دليل [أوليات الخلفية (backend primitives)](/guides/get-started-with-backend-primitives) للمزيد من المعلومات.

## الخيارات
يمكنك تمرير خيارات إلى الأوليات لتخصيصها بحسب احتياجاتك. في حين أن واجهة تابع `run()` للأوليات مشتركة بين جميع التطبيقات، فإن خياراتها تختلف من تطبيق لآخر. راجع مراجع API للتطبيق المحدد للأولية للتعرف على الخيارات التي يدعمها.

على سبيل المثال، راجع موضوعَي [خيارات Estimator](/guides/estimator-options) و[خيارات Sampler](/guides/sampler-options) للتعرف على خيارات أوليات Qiskit Runtime، أو راجع [مراجع واجهة Qiskit Aer API](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html) لخيارات أوليات Qiskit Aer.

## فوائد أوليات Qiskit
بفضل الأوليات، يستطيع مستخدمو Qiskit كتابة كود كمي لـ QPU محدد دون الحاجة إلى إدارة كل التفاصيل بشكل صريح. علاوةً على ذلك، بفضل طبقة التجريد الإضافية، قد تتمكن من الوصول بسهولة أكبر إلى الإمكانات المتقدمة لعتاد مزود معين. فمثلًا، مع أوليات Qiskit Runtime، يمكنك الاستفادة من أحدث التطورات في تخفيف الأخطاء وقمعها بمجرد ضبط خيارات مثل [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level) الخاص بالأولية، بدلًا من بناء تطبيقك الخاص لهذه التقنيات.

بالنسبة لمزودي العتاد، يعني تطبيق الأوليات بشكل أصيل أنه يمكنك تزويد مستخدميك بطريقة أكثر سهولةً "من الصندوق" للوصول إلى ميزات عتادك كتقنيات المعالجة اللاحقة المتقدمة. وهذا يجعل من السهل على مستخدميك الاستفادة من أفضل إمكانات عتادك.

## الخطوات التالية
> **Tip:** - افهم [مدخلات ومخرجات الأوليات](/guides/primitive-input-output).
> - راجع [أمثلة تفصيلية](/guides/simulate-with-qiskit-sdk-primitives).
> - تدرّب على الأوليات من خلال [درس دالة التكلفة](/learning/courses/variational-algorithm-design/cost-functions) في IBM Quantum Learning.
> - راجع [إنشاء مزود](/guides/create-a-provider) لتتعلم كيفية تطبيق أوليات Sampler وEstimator الخاصة بك.
> - اطلع على [مراجع واجهة API](https://docs.quantum.ibm.com/api/qiskit/primitives).
> - اقرأ [الهجرة إلى أوليات V2](/guides/v2-primitives).
> - تعرّف على [أوليات Qiskit Runtime](/guides/qiskit-runtime-primitives)، المستخدمة لتشغيل الدوائر على وحدات QPU من IBM.